# Local Text Embeddings: Step-by-Step
Learn how to generate text embeddings locally and convert sentences into vectors.

## 1) Install dependencies
We will use sentence-transformers to run a local embedding model.

In [ ]:
%pip install sentence-transformers psycopg2-binary

## 2) Load a local embedding model
This uses a small model for fast local inference.

In [6]:
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

## 3) Convert sentences into vectors
The output is a list of numeric vectors (embeddings).

In [7]:


sentences = [
    # "Any places I can find a dog?" , #0
    # "Where can I see dogs in public places?", #0
    "What place has pizza?", #0

    "The dog is on the park" , #1
    "There are puppy on the Plaza" , #2
    "The mall has it's own pizza" , #3
    "The police was patrolling on the road" , #4
    "The water is good in the beach" , #5
    "The it's raining in the forest" , #6
    "There are dogs on the bus" , #7
]


lowerSentence = [s.lower() for s in sentences]

print(lowerSentence)

embeddings = model.encode(lowerSentence)
print(embeddings)
print(embeddings.shape)

['what place has pizza?', 'the dog is on the park', 'there are puppy on the plaza', "the mall has it's own pizza", 'the police was patrolling on the road', 'the water is good in the beach', "the it's raining in the forest", 'there are dogs on the bus']
[[-0.00345207  0.0506988  -0.03276059 ...  0.03641402  0.00505028
  -0.01285796]
 [ 0.02244985 -0.04711957  0.06384178 ...  0.07726322 -0.01975152
  -0.01114014]
 [-0.08915724 -0.02812134  0.03781177 ...  0.10789294  0.02788689
   0.05764466]
 ...
 [-0.00839002 -0.01533703  0.09181897 ... -0.00146245 -0.0009897
   0.07458625]
 [ 0.01014331  0.00412862  0.06011282 ... -0.09143323 -0.0200039
   0.01955765]
 [-0.00027769 -0.03765049  0.03488282 ...  0.15971082  0.00177139
  -0.00101693]]
(8, 384)


## 4) Compare similarity between sentences
Use cosine similarity to see which sentences are closer in meaning.

In [8]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("===========Value===========")
# print("Embeddings: ", embeddings[0])
print("===========================\n")


for index, value in enumerate(embeddings):
    if index != 7:
        result = cosine_similarity(embeddings[index], embeddings[index + 1])
        print(f"Sim 0 - Sim {index + 1} : {result:.2f}")


===========Value===========

Sim 0 - Sim 1 : 0.07
Sim 0 - Sim 2 : 0.36
Sim 0 - Sim 3 : 0.25
Sim 0 - Sim 4 : 0.02
Sim 0 - Sim 5 : 0.05
Sim 0 - Sim 6 : 0.27
Sim 0 - Sim 7 : -0.01


## 5) Embed a single sentence
This is useful for live input or small lookups.

In [9]:
text = "Local embeddings are fast and private."
vector = model.encode([text])[0]
print(vector)
print(len(vector))

[ 2.60942560e-02 -7.29857311e-02  4.50087562e-02 -3.58769787e-03
  1.14214554e-01  8.29491094e-02 -6.04627244e-02 -7.00024143e-02
  4.81808744e-02 -3.40556577e-02  1.24712750e-01  4.68053147e-02
 -1.43438121e-02  2.25963332e-02 -4.69922833e-02 -1.17452245e-03
  8.27708021e-02  3.20072211e-02 -3.39860208e-02  7.03511527e-03
 -7.56127909e-02 -4.57767844e-02  4.65470366e-02 -4.09796536e-02
  1.32990524e-01 -5.89111783e-02 -1.91951524e-02  7.23462701e-02
  1.06314257e-01 -7.02424571e-02  3.43335792e-02  2.20791660e-02
  7.69197149e-03  1.27868783e-02 -4.47853794e-03  9.37895030e-02
 -3.35538201e-02  1.86644644e-02 -1.01269670e-01  1.62323173e-02
  7.37747774e-02  2.08497178e-02 -2.56531853e-02  5.38773043e-03
 -1.63772833e-02 -2.59093679e-02  2.98934653e-02  2.98373085e-02
 -8.33859295e-02 -5.78484591e-03 -1.52942752e-02 -1.41525352e-02
 -9.35616270e-02  6.19795360e-03 -7.22789764e-03 -8.61346647e-02
 -6.86613494e-04 -5.43915406e-02  2.48614568e-02  6.99958652e-02
 -1.96495559e-02 -2.47954

## 6) Database connection
Configure your PostgreSQL connection details.

In [10]:
import os
import psycopg2

# Update these values for your local setup.
db_config = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "llm_iTour"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "moymoybryan"),
}

conn = psycopg2.connect(**db_config)
conn.autocommit = True
cursor = conn.cursor()
print("Connected to PostgreSQL")

Connected to PostgreSQL


## 7) Database generation queries
Create tables, insert seed data, and enable the pgvector extension.

In [ ]:
schema_sql = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS tourist_area (
    area_id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT,
    location TEXT,
    region TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    category TEXT,
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS tourist_area_vectors (
    vector_id SERIAL PRIMARY KEY,
    area_id INTEGER REFERENCES tourist_area(area_id) ON DELETE CASCADE,
    content TEXT,
    embedding VECTOR(1536),
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS area_activities (
    activity_id SERIAL PRIMARY KEY,
    area_id INTEGER REFERENCES tourist_area(area_id) ON DELETE CASCADE,
    activity_name TEXT,
    description TEXT,
    difficulty_level TEXT,
    estimated_duration TEXT,
    price_range TEXT,
    created_at TIMESTAMP WITHOUT TIME ZONE DEFAULT NOW()
);
"""

cursor.execute(schema_sql)
conn.commit()

print("Schema applied")

## 8) Query the database
Run a simple query to verify the data.

In [ ]:
import numpy as np
from data.seed_data import ACTIVITIES, AREAS, VECTORS

areas = AREAS
activities = ACTIVITIES
vectors = VECTORS


def get_or_create_area(area):
    cursor.execute("SELECT area_id FROM tourist_area WHERE name = %s;", (area["name"],))
    row = cursor.fetchone()
    if row:
        return row[0]

    cursor.execute(
        """
        INSERT INTO tourist_area (
            name, description, location, region, latitude, longitude, category
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        RETURNING area_id;
        """,
        (
            area["name"],
            area["description"],
            area["location"],
            area["region"],
            area["latitude"],
            area["longitude"],
            area["category"],
        ),
    )
    return cursor.fetchone()[0]


area_ids = {area["name"]: get_or_create_area(area) for area in areas}

for activity in activities:
    area_id = area_ids[activity["area"]]
    cursor.execute(
        "SELECT 1 FROM area_activities WHERE area_id = %s AND activity_name = %s;",
        (area_id, activity["activity_name"]),
    )
    if cursor.fetchone():
        continue
    cursor.execute(
        """
        INSERT INTO area_activities (
            area_id, activity_name, description, difficulty_level, estimated_duration, price_range
        )
        VALUES (%s, %s, %s, %s, %s, %s);
        """,
        (
            area_id,
            activity["activity_name"],
            activity["description"],
            activity["difficulty_level"],
            activity["estimated_duration"],
            activity["price_range"],
        ),
    )


def normalize_embedding(vec, target_dim=1536):
    if len(vec) == target_dim:
        return vec
    if len(vec) > target_dim:
        return vec[:target_dim]
    # Pad with zeros to match the vector column size.
    return np.pad(vec, (0, target_dim - len(vec)), mode="constant")


def to_vector_literal(vec):
    return "[" + ",".join(f"{x:.6f}" for x in vec) + "]"


for item in vectors:
    area_id = area_ids[item["area"]]
    cursor.execute(
        "SELECT 1 FROM tourist_area_vectors WHERE area_id = %s AND content = %s;",
        (area_id, item["content"]),
    )
    if cursor.fetchone():
        continue

    embedding = model.encode([item["content"]])[0]
    embedding = normalize_embedding(embedding, 1536)
    vector_literal = to_vector_literal(embedding)

    cursor.execute(
        """
        INSERT INTO tourist_area_vectors (area_id, content, embedding)
        VALUES (%s, %s, %s::vector);
        """,
        (area_id, item["content"], vector_literal),
    )

conn.commit()
print("Seed data applied")

## 9) Vector search helper
Define a query filter and embedding search for tourist-related requests.

In [ ]:
show_match_score = False


def is_tourist_query(query: str) -> bool:
    keywords = [
        "tourist",
        "tourism",
        "travel",
        "trip",
        "vacation",
        "itinerary",
        "visit",
        "destination",
        "spot",
        "place",
        "beach",
        "mountain",
        "island",
        "city",
        "museum",
        "park",
        "activity",
        "adventure",
        "food",
        "hotel",
    ]
    query_lower = query.lower()
    return any(word in query_lower for word in keywords)


def normalize_embedding(vec, target_dim=1536):
    if len(vec) == target_dim:
        return vec
    if len(vec) > target_dim:
        return vec[:target_dim]
    return np.pad(vec, (0, target_dim - len(vec)), mode="constant")


def to_vector_literal(vec):
    return "[" + ",".join(f"{x:.6f}" for x in vec) + "]"


def vector_search_tourist(query: str, top_k: int = 5):
    if not is_tourist_query(query):
        print("Query rejected: not related to tourist spots or activities.")
        return []

    query_embedding = model.encode([query])[0]
    query_embedding = normalize_embedding(query_embedding, 1536)
    vector_literal = to_vector_literal(query_embedding)

    cursor.execute(
        """
        SELECT
            ta.name,
            ta.location,
            ta.category,
            tav.content,
            tav.embedding <-> %s::vector AS distance
        FROM tourist_area_vectors tav
        JOIN tourist_area ta ON ta.area_id = tav.area_id
        ORDER BY tav.embedding <-> %s::vector
        LIMIT %s;
        """,
        (vector_literal, vector_literal, top_k),
    )
    rows = cursor.fetchall()

    return rows


user_query = "Any extreme adventures recommend for me?"
vector_search_tourist(user_query)

[('Puerto Princesa Underground River',
  'Palawan',
  'nature',
  'Underground river cave tour and nature park.',
  1.1332413486476163),
 ('Boracay',
  'Aklan',
  'beach',
  'White sand beaches and island hopping tours.',
  1.1512619371654755),
 ('Siargao',
  'Surigao del Norte',
  'beach',
  'Surf breaks, lagoons, and island vibes.',
  1.1756185121169642),
 ('Cebu City',
  'Cebu',
  'urban',
  'Historic downtown, food trips, and museums.',
  1.1857797811823048),
 ('Baguio',
  'Benguet',
  'mountain',
  'Cool climate, pine trees, and viewpoints.',
  1.219583764127745)]

## 10) Generate a tour guide response
Use the matched places to craft a concise, reasoned reply.

In [21]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(model="gemma:2b", temperature=0.2)

def build_reasoned_response(query: str, rows):
    if not rows:
        return (
            "Sorry, I do not have enough data to answer that yet. "
            "Try asking about beaches, mountains, or cultural spots."
        )

    context_lines = [
        f"- {name} in {location}: {content} ({category})"
        for name, location, category, content, distance in rows[:3]
    ]
    context_block = "\n".join(context_lines)

    messages = [
        SystemMessage(content=
            "You are a friendly tour guide. Use the matched places to answer the user's question "
            "and give a short rationale for each suggestion. Keep it concise."
        ),
        HumanMessage(content=(
            f"User question: {query}\n"
            f"Matches:\n{context_block}\n"
            "Answer with up to 3 suggestions and a brief rationale per suggestion. Include estimated price range budgets"
        )),
    ]
    response = llm.invoke(messages)
    return response.content.strip()


user_query = "Any extreme adventures recommend for me?"
rows = vector_search_tourist(user_query)
print("\n" + build_reasoned_response(user_query, rows))


**Suggestion 1: Puerto Princesa Underground River**

* **Nature and adventure:** Explore a unique underground river cave with diverse flora and fauna.
* **Price range:** P10,000 - P15,000 per person.
* **Rationale:** The cave tour offers a thrilling and unique experience unlike any other in the world.

**Suggestion 2: Boracay**

* **Beach paradise:** Relax on pristine white sand beaches and enjoy island hopping tours.
* **Price range:** P15,000 - P20,000 per person.
* **Rationale:** Boracay is a haven for beach lovers, offering endless opportunities for swimming, sunbathing, and water activities.

**Suggestion 3: Siargao**

* **Surfing and adventure:** Enjoy world-class surfing breaks, explore lagoons, and experience island life.
* **Price range:** P20,000 - P25,000 per person.
* **Rationale:** Siargao offers a perfect blend of adventure and relaxation, with ample opportunities for surfing, swimming, and island exploration.
